In [ ]:
!pip install torch torchvision onnx onnxruntime

In [ ]:
!pip install onnx onnxruntime onnxscript

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import convnext_tiny
import onnxruntime as ort
import torch
from PIL import Image
from torchvision.models import ConvNeXt_Tiny_Weights
from torchvision import transforms
import numpy as np
import torch.nn.functional as F
from google.colab import files

In [ ]:
# upload the convnext weights
uploaded = files.upload()

Saving Convnext_Best_AllDataset(0.660).pt to Convnext_Best_AllDataset(0.660) (1).pt


In [ ]:
# ---------------------------------------------------
# Recreate the model architecture
# We rebuild the same ConvNeXt model structure
# that was used during training.
# ---------------------------------------------------
NUM_CLASSES = 5

model = convnext_tiny(weights=None)

in_features = model.classifier[2].in_features
model.classifier[2] = nn.Sequential(
    nn.Dropout(0.05),
    nn.Linear(in_features, NUM_CLASSES)
)
# ---------------------------------------------------
# Load the trained weights from the .pt file
# This restores the learned parameters of the model.
# ---------------------------------------------------

state_dict = torch.load("Convnext_Best_AllDataset(0.660).pt", map_location="cpu")
model.load_state_dict(state_dict)

# ---------------------------------------------------
# Set the model to evaluation mode
# This disables training behaviors like dropout
# and ensures correct inference behavior.
# ---------------------------------------------------
model.eval()
print("Model loaded successfully")

Model loaded successfully


In [ ]:
# ---------------------------------------------------
# Create a dummy input tensor
# ONNX export requires an example input to trace
# the computation graph of the model.
#
# Shape explanation:
# (batch_size, channels, height, width)
# ConvNeXt expects 224x224 RGB images.
# ---------------------------------------------------

dummy_input = torch.randn(1, 3, 224, 224)

In [ ]:
# ---------------------------------------------------
# Export the PyTorch model to ONNX format
# This converts the computation graph and weights
# into a framework-independent format.
# ---------------------------------------------------
torch.onnx.export(
    model,
    dummy_input,
    "bouh_convnext_classifier_model.onnx",
    input_names=["image"],
    output_names=["logits"],

    # ------------------------------------------------
    # dynamic_axes allows variable batch sizes
    # This is required for dynamic batching
    # when using inference servers.
    # ------------------------------------------------
    dynamic_axes={
        "image": {0: "batch_size"},
        "logits": {0: "batch_size"}
    },

    opset_version=13,
    do_constant_folding=True,
    dynamo=False
    )

print("ONNX model exported successfully")

files.download("bouh_convnext_classifier_model.onnx")

/tmp/ipykernel_40460/1441069668.py:6: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX model exported successfully


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import onnx

model = onnx.load("bouh_convnext_classifier_model.onnx")
print("IR version:", model.ir_version)
print("Opset:", model.opset_import[0].version)

IR version: 7
Opset: 13


In [ ]:
# ---------------------------------------------------
# 7. Open the uploaded image
# ---------------------------------------------------
uploaded = files.upload()
image = Image.open("anxiety.png").convert("RGB")


# ---------------------------------------------------
# 8. Apply the same preprocessing used during training
# ConvNeXt expects images normalized with ImageNet stats
# ---------------------------------------------------
weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
mean, std = weights.transforms().mean, weights.transforms().std

transform = transform = transforms.Compose([
    transforms.Resize(int(224*1.14)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

input_tensor = transform(image).unsqueeze(0)

Saving anxiety.png to anxiety.png


In [ ]:
import onnx
import onnxruntime as ort
import torch

# ---------------------------------------------------
# PyTorch inference
# ---------------------------------------------------
with torch.no_grad():
    pytorch_output = model(input_tensor)
# ---------------------------------------------------
# ONNX inference
# ---------------------------------------------------
session = ort.InferenceSession("bouh_convnext_classifier_model.onnx")

onnx_output = session.run(
    None,
    {"image": input_tensor.numpy()}
)

In [ ]:
# ---------------------------------------------------
# class mapping used during training
# ---------------------------------------------------

CLASSES = [
    "تفاؤل",
    "حزن",
    "غضب",
    "خوف",
    "توتر وقلق"
]

# label → index
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

# index → label
idx_to_class = {i: c for c, i in class_to_idx.items()}

print(idx_to_class)

{0: 'تفاؤل', 1: 'حزن', 2: 'غضب', 3: 'خوف', 4: 'توتر وقلق'}


In [ ]:
# verify correct format transformation

# convert logits → probabilities
pytorch_probs = F.softmax(pytorch_output, dim=1).numpy()

# fix ONNX tensor shape
onnx_logits = onnx_output[0]
onnx_probs = F.softmax(torch.tensor(onnx_logits), dim=1).numpy()

# predicted classes
pytorch_pred = np.argmax(pytorch_probs)
onnx_pred = np.argmax(onnx_probs)

print("PyTorch prediction index:", pytorch_pred)
print("ONNX prediction index:", onnx_pred)

print("PyTorch class:", idx_to_class[pytorch_pred])
print("ONNX class:", idx_to_class[onnx_pred])

print("\nClass probabilities:")
print("--------------------")

for i in range(len(idx_to_class)):
    print(
        "Index:", i,
        "| Class:", idx_to_class[i],
        "| PyTorch:", round(float(pytorch_probs[0][i]),4),
        "| ONNX:", round(float(onnx_probs[0][i]),4)
    )

PyTorch prediction index: 4
ONNX prediction index: 4
PyTorch class: توتر وقلق
ONNX class: توتر وقلق

Class probabilities:
--------------------
Index: 0 | Class: تفاؤل | PyTorch: 0.0236 | ONNX: 0.0236
Index: 1 | Class: حزن | PyTorch: 0.0172 | ONNX: 0.0172
Index: 2 | Class: غضب | PyTorch: 0.4342 | ONNX: 0.4342
Index: 3 | Class: خوف | PyTorch: 0.0037 | ONNX: 0.0037
Index: 4 | Class: توتر وقلق | PyTorch: 0.5213 | ONNX: 0.5213


In [ ]:
# convert tensors to numpy
pytorch_logits = pytorch_output.numpy()
onnx_logits = onnx_output[0]

# compute difference
delta = pytorch_logits - onnx_logits

print("PyTorch logits:", pytorch_logits)
print("ONNX logits:", onnx_logits)

print("\nDelta (PyTorch - ONNX):")
print(delta)

# format number instead of scientific notation
max_diff = np.max(np.abs(delta))
print(f"\nMax absolute difference: {max_diff:.6f}")

PyTorch logits: [[-0.8735494 -1.1916527  2.0385659 -2.7188041  2.2214427]]
ONNX logits: [[-0.8735479 -1.191654   2.038566  -2.7188065  2.2214468]]

Delta (PyTorch - ONNX):
[[-1.4901161e-06  1.3113022e-06 -2.3841858e-07  2.3841858e-06
  -4.0531158e-06]]

Max absolute difference: 0.000004
